# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [1]:
from model.Model import model_MSG 
from model.Plotter import plotter_MSG
import os
import sys


# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

In [2]:
model_MSG.clone_scenario

<function model.message_phases.model_MSG.clone_scenario(self, scenario_name, keep_solution=False) -> tuple>

In [2]:
# Sys and OS Operations

os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [ ]:
# Set-up

msg = model_MSG(scenario_name='baseline')
platform, msg_scenario, msg_folder = msg.setup_scenario()

Config file read successfully!
####################################################################################################
Config file 'model/scenarios\baseline\config.toml' passed format verification!
####################################################################################################

		Model creation time: 6 Min and 57 Sec 		


In [7]:
output_path="C:/Users/brend/Documents/Github/message-ix - CA/scenarios/baseline/MESSAGE_test.csv" 

In [73]:
import pandas as pd
import pyam as pyam
import message_ix as message_ix

In [11]:
def generate_report(
    msg,
    output_path="C:/Users/brend/Documents/Github/message-ix - CA/scenarios/baseline/MESSAGE_test.csv",
    min_year= 2015,
    max_year= 2060,
    from_excel=False,
    unit_correct=True,
 ):
    """
    Generate a report from the MESSAGEix scenario and save it to a CSV file.

    Args:
                    output_path (str): The path where the output CSV file will be saved.
                    min_year (int): The minimum year for the report.
                    max_year (int): The maximum year for the report.
                    from_excel (bool): Whether to read the report from an Excel file.
                    unit_correct (bool): Whether to correct units in the report.
    """
    rep = message_ix.Reporter.from_scenario(
        msg.scenario,
        units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
    )
    report = rep.get("message::default")

    if from_excel:
        report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
    else:
        report_df = report.timeseries()
        report_df.reset_index(inplace=True)
        report_df.columns = report_df.columns.astype(str)
        report_df.columns = report_df.columns.str.title()

    years = msg.scenario.set("year").tolist()
    years = [str(x) for x in years if x >= min_year and x <= max_year]

    report_df.fillna(0, inplace=True)

    report_df.columns = report_df.columns.astype(str)
    report_df = report_df.drop(
        report_df.columns.to_series()["2020" : str(min_year)], axis=1
    )

    in_flows = report_df[report_df["Variable"].str.startswith("in|")]
    out_flows = report_df[report_df["Variable"].str.startswith("out|")]
    all_flows = pd.concat([out_flows, in_flows])

    report_df["Model"] = "MESSAGEix-Canada"
    report_df["Scenario"] = "Reference"
    report_df["Variable"] = report_df["Variable"].str.replace(
        "emis", "Emissions", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "in|final|", "Final Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "out|final|", "Final Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "in|primary|", "Primary Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "out|primary|", "Primary Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "in|secondary|", "Secondary Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "out|secondary|", "Secondary Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "in|renewable|", "Primary Energy|renewable|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "out|renewable|", "Primary Energy|renewable|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "total om cost|", "Total Costs|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "out|useful|", "Useful Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "in|useful|", "Useful Energy|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "CAP|capacity|", "Capacity|", regex=False
    )
    report_df["Variable"] = report_df["Variable"].str.replace(
        "inv cost|", "Capital Cost|", regex=False
    )
    report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

    pyam_df = pyam.IamDataFrame(report_df)

    variable = pyam_df.variable
    pyam_df.aggregate_region(variable, append=True)
    pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

    pyam_df.to_csv(output_path)

    print(f"Report saved to {output_path}")

In [49]:
report_df.index.names

FrozenList([None])

In [74]:
output_path="C:/Users/brend/Documents/Github/message-ix - CA/scenarios/baseline/MESSAGE_test_updated_emission.csv"
min_year= 2015
max_year= 2060
from_excel=False
unit_correct=True

rep = message_ix.Reporter.from_scenario(
    msg.scenario,
    units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
)
report = rep.get("message::default")

if from_excel:
    report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
else:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

years = msg.scenario.set("year").tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

report_df.fillna(0, inplace=True)

report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(
    report_df.columns.to_series()["2020" : str(min_year)], axis=1
)

in_flows = report_df[report_df["Variable"].str.startswith("in|")]
out_flows = report_df[report_df["Variable"].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Model"] = "MESSAGEix-Canada"
report_df["Scenario"] = "Reference"
report_df["Variable"] = report_df["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

report_df.columns = report_df.columns.str.strip()  # Remove any leading/trailing whitespace
report_df.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_df = report_df.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_df = pyam.IamDataFrame(report_df)

variable = pyam_df.variable
pyam_df.aggregate_region(variable, append=True)
pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

pyam_df.to_csv(output_path)

print(f"Report saved to {output_path}")



at.ac.iiasa.ixmp.exceptions.IxException: at.ac.iiasa.ixmp.exceptions.IxException: There was a problem checking for a blob of item ACT_LO in the database!

In [69]:
import os

output_dir = r'C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline'

# Check if the directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'report.csv')

# Now save the file
pyam_df.to_csv(output_path)
print(f"Report saved to {output_path}")

Report saved to C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline\report.csv


In [52]:
print(report_df["Region"].value_counts())

Region
Alberta                    3355
BritishColumbia            3355
Manitoba                   3355
NewBrunswick               3355
NewfoundlandandLabrador    3355
NorthwestTerritories       3355
NovaScotia                 3355
Nunavut                    3355
Ontario                    3355
PrinceEdwardIsland         3355
Quebec                     3355
Saskatchewan               3355
Yukon                      3355
Name: count, dtype: int64


In [59]:
report_df[report_df["variable"] =='Secondary Energy|electr|h2_smr_ccs|M1']


,model,scenario,Region,variable,Unit,2010,2015,2020,2025,2030,2035,2040,2045,2050,2055,2060
2816,MESSAGEix-Canada,Reference,Alberta,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3194,MESSAGEix-Canada,Reference,Alberta,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6171,MESSAGEix-Canada,Reference,BritishColumbia,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6549,MESSAGEix-Canada,Reference,BritishColumbia,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9526,MESSAGEix-Canada,Reference,Manitoba,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9904,MESSAGEix-Canada,Reference,Manitoba,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12881,MESSAGEix-Canada,Reference,NewBrunswick,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13259,MESSAGEix-Canada,Reference,NewBrunswick,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16236,MESSAGEix-Canada,Reference,NewfoundlandandLabrador,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16614,MESSAGEix-Canada,Reference,NewfoundlandandLabrador,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
print(report_df["variable"].value_counts())

variable
Secondary Energy|electr|h2_smr_ccs|M1                             26
Secondary Energy|electr|h2_coal|M1                                26
Secondary Energy|electr|h2_bio|M1                                 26
Secondary Energy|electr|h2_bio_ccs|M1                             26
Secondary Energy|electr|h2_smr|M1                                 26
                                                                  ..
Emissions|SO2|eth_rc|M1                                           13
Emissions|SO2|flaring_CO2|M1                                      13
Emissions|SO2|foil_hpl|M1                                         13
Emissions|SO2|foil_i|M1                                           13
out|water_supply|upstream_landuse|extract__upstream_landuse|M1    13
Name: count, Length: 3342, dtype: int64


In [30]:
report_df.columns

Index(['Model', 'Scenario', 'Region', 'Variable', 'Unit', '2010', '2015',
       '2020', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060'],
      dtype='object')

In [38]:
report_df.head()

,Model,Scenario,Region,Variable,Unit,2010,2015,2020,2025,2030,2035,2040,2045,2050,2055,2060
0,MESSAGEix-Canada,Reference,Alberta,Capacity Addition|LNG_exp,,0.0,0.0,0.067645,0.109214,0.176326,0.28468,0.459617,0.8097,1.307267,2.110592,3.407566
1,MESSAGEix-Canada,Reference,Alberta,Capacity Addition|adipic_thermal,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
2,MESSAGEix-Canada,Reference,Alberta,Capacity Addition|ammonia_secloop,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
3,MESSAGEix-Canada,Reference,Alberta,Capacity Addition|bio_hpl,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000
4,MESSAGEix-Canada,Reference,Alberta,Capacity Addition|bio_hpl__air,,0.0,0.0,0.000000,0.000000,0.000000,0.00000,0.000000,0.0000,0.000000,0.000000,0.000000


In [64]:
test = report_df['variable'].value_counts()
# print("Number of unique variables:", report_df['Variable'].nunique())


variable
Secondary Energy|electr|h2_smr_ccs|M1                             26
Secondary Energy|electr|h2_coal|M1                                26
Secondary Energy|electr|h2_bio|M1                                 26
Secondary Energy|electr|h2_bio_ccs|M1                             26
Secondary Energy|electr|h2_smr|M1                                 26
                                                                  ..
Emissions|SO2|eth_rc|M1                                           13
Emissions|SO2|flaring_CO2|M1                                      13
Emissions|SO2|foil_hpl|M1                                         13
Emissions|SO2|foil_i|M1                                           13
out|water_supply|upstream_landuse|extract__upstream_landuse|M1    13
Name: count, Length: 3342, dtype: int64

In [12]:
generate_report(msg)

var_cost: mixed units ['USD/kW', 'USD/kWa'] discarded
emission_factor: mixed units ['tC', '???', 'kg/kWa', '-'] discarded
input: mixed units ['GWa', '-', '%'] discarded
output: mixed units ['GWa', '-'] discarded
fix_cost: mixed units ['USD/kW', 'USD/kWa'] discarded
inv_cost: mixed units ['USD/kW', 'USD/kWa'] discarded


Duplicate Rows:
                   Model   Scenario   Region  \
3180   MESSAGEix-Canada  Reference  Alberta   
3190   MESSAGEix-Canada  Reference  Alberta   
3191   MESSAGEix-Canada  Reference  Alberta   
3192   MESSAGEix-Canada  Reference  Alberta   
3193   MESSAGEix-Canada  Reference  Alberta   
...                 ...        ...      ...   
43464  MESSAGEix-Canada  Reference    Yukon   
43465  MESSAGEix-Canada  Reference    Yukon   
43466  MESSAGEix-Canada  Reference    Yukon   
43467  MESSAGEix-Canada  Reference    Yukon   
43510  MESSAGEix-Canada  Reference    Yukon   

                                       Variable Unit  2010  2015  2020  2025  \
3180     Secondary Energy|electr|eth_bio_ccs|M1        0.0   0.0   0.0   0.0   
3190      Secondary Energy|electr|h2_bio_ccs|M1        0.0   0.0   0.0   0.0   
3191          Secondary Energy|electr|h2_bio|M1        0.0   0.0   0.0   0.0   
3192     Secondary Energy|electr|h2_coal_ccs|M1        0.0   0.0   0.0   0.0   
3193         Secon

ValueError: Duplicate rows in `data`:
              model   scenario   region  \
0  MESSAGEix-Canada  Reference  Alberta   
1  MESSAGEix-Canada  Reference  Alberta   
2  MESSAGEix-Canada  Reference  Alberta   
3  MESSAGEix-Canada  Reference  Alberta   
4  MESSAGEix-Canada  Reference  Alberta   

                                 variable unit  year  
0  Secondary Energy|electr|eth_bio_ccs|M1       2010  
1  Secondary Energy|electr|eth_bio_ccs|M1       2015  
2  Secondary Energy|electr|eth_bio_ccs|M1       2020  
3  Secondary Energy|electr|eth_bio_ccs|M1       2025  
4  Secondary Energy|electr|eth_bio_ccs|M1       2030  
...

NameError: name 'report_df' is not defined

In [6]:
#from message_ix_models.tools.costs.config import Config
#from message_ix_models.tools.costs.projections import create_cost_projections

In [ ]:
'''# Step 1: Define hydrogen-related technologies
hydrogen_tech_list = ["h2_atr", "h2_atr_ccs"]

# Step 2: Define configuration for cost projections using the 'convergence' method
cfg = Config(
    module="energy",  # Assuming hydrogen falls under energy module
    method="convergence",  # Use convergence method for cost projections
    base_year=2020,  # Base year for the cost projections
    convergence_year=2050,  # Year to which costs should converge
    node="R12",  # Set your region, e.g., R12 (or your specific region code)
    ref_region="R12_NAM"  # Set the reference region
)

# Step 3: Generate base cost projections for all energy technologies
cost_projections = create_cost_projections(cfg)

# Step 4: Extract investment and fixed cost DataFrames
inv_cost_df = cost_projections["inv_cost"]
fix_cost_df = cost_projections["fix_cost"]

# Print columns for debugging
print("Investment Cost DataFrame Columns:")
print(inv_cost_df.columns)

print("\nFixed Cost DataFrame Columns:")
print(fix_cost_df.columns)

# Step 5: Filter cost projections for hydrogen-related technologies
hydrogen_inv_cost = inv_cost_df[inv_cost_df['technology'].isin(hydrogen_tech_list)]

# Debugging: Print the filtered DataFrame to understand its structure
print("\nFiltered Hydrogen Investment Cost DataFrame:")
print(hydrogen_inv_cost.head())

# Check if 'year' column is present
if 'year' not in hydrogen_inv_cost.columns:
    print("No 'year' column found in hydrogen investment costs. Available columns are:", hydrogen_inv_cost.columns)

# Step 6: Apply the learning rate manually to the investment cost
learning_rate = 0.1  # 10% learning rate

# Use 'year' instead of 'year_act' for base year costs
base_year_costs = hydrogen_inv_cost[hydrogen_inv_cost['year'] == 2020].set_index('technology')['inv_cost']

# Apply learning rate decay to future years
for year in hydrogen_inv_cost['year'].unique():
    if year > 2020:
        for tech in hydrogen_tech_list:
            # Calculate the new cost for each technology
            if tech in base_year_costs.index:
                new_cost = base_year_costs[tech] * (1 - learning_rate) ** ((year - 2020) / 5)
                # Update the DataFrame
                hydrogen_inv_cost.loc[
                    (hydrogen_inv_cost['year'] == year) & (hydrogen_inv_cost['technology'] == tech), 'inv_cost'] = new_cost

# Step 7: Extract final investment and fixed costs
inv_cost = hydrogen_inv_cost[['technology', 'year', 'inv_cost']] if 'inv_cost' in hydrogen_inv_cost.columns else None
fix_cost = fix_cost_df[fix_cost_df['technology'].isin(hydrogen_tech_list)] if 'technology' in fix_cost_df.columns else None

# You can now apply these adjusted costs to your MESSAGEix model.

# Print adjusted costs for verification
print("\nAdjusted Investment Costs:")
print(inv_cost)'''

In [5]:
msg.scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [18]:
msg_plt = Plotter_MSG(msg_scenario)
nodes = msg_scenario.set('node')[2:]


In [ ]:
msg_plt.plotter(case="test",nodeloc=nodes,path="output",yr_min=2020, yr_max=2060)